# Document-Grounded ChatBot
### Python · LangChain · Gemini

| Step | What happens |
|------|--------------|
| 1 | Load a document |
| 2 | Split it into chunks |
| 3 | Convert chunks into embeddings and store them |
| 4 | For each question, retrieve the most relevant chunks |
| 5 | Send chunks + chat history + question to Gemini |
| 6 | Answer **only** from the document — say *"I don't have that information"* otherwise |

> **Key design goal:** the bot must never answer from outside the loaded document.

---
## 0 · Install dependencies

In [ ]:
%pip install -q \
    langchain \
    langchain-google-genai \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pypdf \
    python-dotenv

---
## 0b · Configure API key

In [ ]:
import os, warnings, time
warnings.filterwarnings('ignore')

GOOGLE_API_KEY = 'AQ.Ab8RN6KubRtdY-jkA8ESORSKdwxx3Hw5Oc62Y7ExRcQxhXNSIw'
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print('API key set.')

---
## Step 1 · Load a document

Supports `.txt` and `.pdf`. Change `DOC_PATH` to your own file.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader, PyPDFLoader

DOC_PATH = 'sample_doc.txt'   # change to your .txt or .pdf

def load_document(path):
    ext = Path(path).suffix.lower()
    loader = PyPDFLoader(path) if ext == '.pdf' else TextLoader(path, encoding='utf-8')
    docs = loader.load()
    print(f'Loaded {len(docs)} page(s) — {sum(len(d.page_content) for d in docs)} chars')
    return docs

raw_docs = load_document(DOC_PATH)
print(f'\nPreview:\n{raw_docs[0].page_content[:300]}')

---
## Step 2 · Split into chunks

| Parameter | Value | Why |
|-----------|-------|-----|
| `chunk_size` | 800 chars | Fits embedding model limits |
| `chunk_overlap` | 150 chars | Prevents answers from being cut across boundaries |

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', ''],
).split_documents(raw_docs)

print(f'{len(chunks)} chunks created:')
for i, c in enumerate(chunks, 1):
    print(f'  [{i}] {len(c.page_content)} chars — {c.page_content[:70].strip()}...')

---
## Step 3 · Embeddings → FAISS vector store

Each chunk is converted to a dense vector using **Gemini Embedding 2** and stored in a local **FAISS** index.  
FAISS runs entirely in-memory — no external database needed.

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2')

print('Embedding chunks...')
vector_store = FAISS.from_documents(chunks, embeddings)
print(f'Done. {vector_store.index.ntotal} vectors in FAISS index.')

---
## Step 4 · Retriever

For each question the retriever embeds the query and returns the `k=4` most similar chunks.

In [ ]:
import time

retriever = vector_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3},
)

# Retry the first query — Gemini embedding API sometimes needs a moment
# after bulk indexing before accepting new requests (avoids 500 errors)
for attempt in range(3):
    try:
        test_results = retriever.invoke('What is the annual leave policy?')
        break
    except Exception as e:
        if attempt < 2:
            print(f'Attempt {attempt+1} failed ({e}), retrying in 2s...')
            time.sleep(2)
        else:
            raise

print(f'Retriever test — {len(test_results)} chunks returned:')
for i, doc in enumerate(test_results, 1):
    print(f'  [{i}] {doc.page_content[:120].strip()}...')

---
## Steps 5 & 6 · LLM + strict document-only prompt

The system prompt is the critical safety gate:
- The model receives **only** the retrieved chunks as its knowledge source.
- It is explicitly instructed to say **"I don't have that information"** when the answer is absent.
- `temperature=0` makes the output deterministic, reducing hallucination risk.

In [ ]:
import time
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatGoogleGenerativeAI(model='models/gemini-3.6-flash', temperature=0)

qa_system_prompt = """You are a document assistant. Answer ONLY from the document excerpts in <context>.

Rules - no exceptions:
1. Base your answer exclusively on <context>.
2. Do NOT use training knowledge or outside information.
3. If the answer is not in <context>, respond with exactly: "I don't have that information."
4. Do not guess or extrapolate beyond what is explicitly stated.

<context>
{context}
</context>"""

qa_prompt = ChatPromptTemplate.from_messages([
    ('system', qa_system_prompt),
    MessagesPlaceholder('chat_history'),
    ('human', '{question}'),
])

answer_chain = qa_prompt | llm | StrOutputParser()

def format_docs(docs):
    return '\n\n---\n\n'.join(d.page_content for d in docs)

def build_retrieval_query(question, chat_history):
    if not chat_history:
        return question
    last_human = next((m.content for m in reversed(chat_history) if isinstance(m, HumanMessage)), '')
    return f"{last_human} {question}" if last_human else question

def call_with_retry(fn, payload, label='call', retries=3, delay=2):
    for attempt in range(retries):
        try:
            return fn(payload)
        except Exception as e:
            if attempt < retries - 1:
                wait = delay * (attempt + 1)
                print(f'[{label} retry {attempt+1}] {e} — waiting {wait}s...')
                time.sleep(wait)
            else:
                raise

def ask(question, chat_history=[]):
    """Returns full answer string with retry on both retrieval and LLM."""
    retrieval_query = build_retrieval_query(question, chat_history)
    docs    = call_with_retry(retriever.invoke, retrieval_query, label='retrieval')
    context = format_docs(docs)
    payload = {'question': question, 'chat_history': chat_history, 'context': context}
    return call_with_retry(answer_chain.invoke, payload, label='LLM')

print('Chain ready.')

---
## Chat — interactive session

Type your questions. Type `quit` to stop.

In [ ]:
chat_history = []

print('DocBot ready. Ask anything about the loaded document.')
print("Type 'quit' to exit.")
print('=' * 60)

while True:
    question = input('\nYou: ').strip()
    if not question:
        continue
    if question.lower() in {'quit', 'exit'}:
        print('Goodbye!')
        break

    answer = ask(question, chat_history)
    print(f'\nBot: {answer}')
    print('-' * 60)

    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

---
## Debug — inspect retrieved chunks + answer for any question

In [ ]:
def inspect(question):
    docs    = retriever.invoke(question)
    context = format_docs(docs)
    answer  = answer_chain.invoke({'question': question, 'chat_history': [], 'context': context})
    print(f'Q: {question}')
    print('=' * 60)
    print('Retrieved chunks:')
    for i, doc in enumerate(docs, 1):
        print(f'  [{i}] {doc.page_content.strip()[:200]}')
    print('-' * 60)
    print(f'Answer: {answer}')

# In-document question
inspect('How many days of annual leave do employees get?')

In [ ]:
# Out-of-document question — bot must say "I don't have that information."
inspect('What is the company stock price?')